<a href="https://colab.research.google.com/github/AnisaML07/chronicroisk-ke/blob/main/notebooks/01_data_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ChronicRisk KE — Data Cleaning Notebook
### Phase 1 of 4 | CRISP-DM: Data Preparation

This notebook loads the cleaned KDHS 2022 dataset and prepares it for machine learning.

**Steps covered:**
1. Import libraries
2. Load the dataset
3. Explore the data
4. Handle missing values
5. Fix data types
6. Remove outliers
7. Save the final cleaned dataset

In [4]:
# ── Import all libraries needed for data cleaning ──────────────────────
import pandas as pd        # for loading and manipulating data
import numpy as np         # for numerical operations
import matplotlib.pyplot as plt  # for charts
import seaborn as sns      # for nicer charts

# Display settings
pd.set_option('display.max_columns', None)  # show all columns
pd.set_option('display.float_format', '{:.2f}'.format)  # 2 decimal places

print("✅ Libraries imported successfully")

✅ Libraries imported successfully


In [6]:
# ── Load the dataset ────────────────────────────────────────────────────
df = pd.read_csv('ChronicRiskKE_clean.csv')

# First look
print(f"✅ Dataset loaded successfully")
print(f"   Rows    : {len(df):,}")
print(f"   Columns : {len(df.columns)}")
print(f"\nColumn names:")
print(list(df.columns))

✅ Dataset loaded successfully
   Rows    : 46,609
   Columns : 14

Column names:
['gender', 'age', 'county', 'residence', 'education', 'wealth', 'weight_kg', 'height_cm', 'bmi', 'smokes', 'alcohol_days', 'hypertension', 'diabetes', 'heart_disease']


In [7]:
# ── Preview the first 5 rows ────────────────────────────────────────────
df.head()

,gender,age,county,residence,education,wealth,weight_kg,height_cm,bmi,smokes,alcohol_days,hypertension,diabetes,heart_disease
0,Female,34,Mombasa,Urban,No education,Richer,97.30,161.40,37.33,No,Never drinks,0.00,0.00,0.00
1,Female,38,Mombasa,Urban,Secondary,Richest,NaN,NaN,NaN,NaN,Unknown,NaN,NaN,NaN
2,Female,33,Mombasa,Urban,Primary,Richest,97.60,165.20,35.74,No,Never drinks,0.00,0.00,0.00
3,Female,39,Mombasa,Urban,Secondary,Richest,102.80,161.30,39.49,No,1 days,0.00,0.00,0.00
4,Female,30,Mombasa,Urban,Primary,Richer,50.80,159.10,20.07,No,Never drinks,0.00,0.00,0.00


In [8]:
# ── Check data types and non-null counts ────────────────────────────────
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46609 entries, 0 to 46608
Data columns (total 14 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   gender         46609 non-null  object 
 1   age            46609 non-null  int64  
 2   county         46609 non-null  object 
 3   residence      46609 non-null  object 
 4   education      46609 non-null  object 
 5   wealth         46609 non-null  object 
 6   weight_kg      16901 non-null  float64
 7   height_cm      16901 non-null  float64
 8   bmi            16676 non-null  float64
 9   smokes         16902 non-null  object 
 10  alcohol_days   46609 non-null  object 
 11  hypertension   31354 non-null  float64
 12  diabetes       31354 non-null  float64
 13  heart_disease  16901 non-null  float64
dtypes: float64(6), int64(1), object(7)
memory usage: 5.0+ MB


In [9]:
# ── Check missing values across all columns ─────────────────────────────
missing = pd.DataFrame({
    'Missing Count': df.isnull().sum(),
    'Missing %': (df.isnull().sum() / len(df) * 100).round(2)
})

print("=== MISSING VALUES REPORT ===\n")
print(missing)

=== MISSING VALUES REPORT ===

               Missing Count  Missing %
gender                     0       0.00
age                        0       0.00
county                     0       0.00
residence                  0       0.00
education                  0       0.00
wealth                     0       0.00
weight_kg              29708      63.74
height_cm              29708      63.74
bmi                    29933      64.22
smokes                 29707      63.74
alcohol_days               0       0.00
hypertension           15255      32.73
diabetes               15255      32.73
heart_disease          29708      63.74


In [10]:
# ── Step 1: Drop rows where target variables are missing ─────────────────
# We cannot train a model without knowing the outcome
df = df.dropna(subset=['hypertension', 'diabetes'])

print(f"✅ Rows after dropping missing targets: {len(df):,}")
print(f"   Removed: {46609 - len(df):,} rows")
print(f"\nHypertension cases: {df['hypertension'].value_counts().to_dict()}")
print(f"Diabetes cases    : {df['diabetes'].value_counts().to_dict()}")

✅ Rows after dropping missing targets: 31,354
   Removed: 15,255 rows

Hypertension cases: {0.0: 29551, 1.0: 1803}
Diabetes cases    : {0.0: 31082, 1.0: 272}


In [11]:
# ── Step 2: Drop columns we don't need ──────────────────────────────────
# weight_kg and height_cm are already captured by BMI
# heart_disease has 63% missing and is not our prediction target

df = df.drop(columns=['weight_kg', 'height_cm', 'heart_disease'])

print(f"✅ Columns after dropping: {list(df.columns)}")
print(f"   Total columns remaining: {len(df.columns)}")

✅ Columns after dropping: ['gender', 'age', 'county', 'residence', 'education', 'wealth', 'bmi', 'smokes', 'alcohol_days', 'hypertension', 'diabetes']
   Total columns remaining: 11


In [12]:
# ── Step 3: Fill missing BMI with the median ─────────────────────────────
# Median is used instead of mean because BMI can have outliers
# that would skew the mean

bmi_median = df['bmi'].median()
df['bmi'] = df['bmi'].fillna(bmi_median)

print(f"✅ BMI missing values filled with median: {bmi_median:.2f}")
print(f"   BMI missing values remaining: {df['bmi'].isnull().sum()}")

✅ BMI missing values filled with median: 22.81
   BMI missing values remaining: 0


In [13]:
# ── Step 4: Fill missing smoking status with 'Unknown' ───────────────────
# These are male respondents whose smoking data wasn't in the men's recode
# 'Unknown' is an honest label — we don't assume they don't smoke

df['smokes'] = df['smokes'].fillna('Unknown')

print(f"✅ Smoking values filled")
print(f"   Smoking categories: {df['smokes'].value_counts().to_dict()}")

✅ Smoking values filled
   Smoking categories: {'No': 16795, 'Unknown': 14453, 'Yes': 106}


In [14]:
# ── Step 5: Convert target columns from float to integer ─────────────────
# They currently show as 0.0 and 1.0 — should be 0 and 1

df['hypertension'] = df['hypertension'].astype(int)
df['diabetes']     = df['diabetes'].astype(int)
df['age']          = df['age'].astype(int)

print(f"✅ Data types fixed")
print(f"\nUpdated data types:")
print(df.dtypes)

✅ Data types fixed

Updated data types:
gender           object
age               int64
county           object
residence        object
education        object
wealth           object
bmi             float64
smokes           object
alcohol_days     object
hypertension      int64
diabetes          int64
dtype: object


In [15]:
# ── Step 6: Check for duplicate rows ────────────────────────────────────
duplicates = df.duplicated().sum()
print(f"Duplicate rows found: {duplicates}")

# Remove if any exist
if duplicates > 0:
    df = df.drop_duplicates()
    print(f"✅ Duplicates removed. Rows remaining: {len(df):,}")
else:
    print(f"✅ No duplicates found. All {len(df):,} rows are unique")

Duplicate rows found: 2857
✅ Duplicates removed. Rows remaining: 28,497


In [16]:
# ── Step 7: Remove BMI outliers ──────────────────────────────────────────
# WHO defines healthy BMI range as 10–60 for adults
# Values outside this range are likely data entry errors

print(f"BMI statistics before cleaning:")
print(df['bmi'].describe())

before = len(df)
df = df[(df['bmi'] >= 10) & (df['bmi'] <= 60)]
after = len(df)

print(f"\n✅ BMI outliers removed")
print(f"   Rows before : {before:,}")
print(f"   Rows after  : {after:,}")
print(f"   Removed     : {before - after:,} rows")
print(f"\nBMI statistics after cleaning:")
print(df['bmi'].describe())

BMI statistics before cleaning:
count   28497.00
mean       23.42
std         4.09
min        12.04
25%        21.98
50%        22.81
75%        23.77
max        99.98
Name: bmi, dtype: float64

✅ BMI outliers removed
   Rows before : 28,497
   Rows after  : 28,494
   Removed     : 3 rows

BMI statistics after cleaning:
count   28494.00
mean       23.41
std         4.01
min        12.04
25%        21.98
50%        22.81
75%        23.77
max        59.91
Name: bmi, dtype: float64


In [17]:
# ── Step 8: Remove age outliers ──────────────────────────────────────────
# KDHS surveys adults aged 15-54 (women) and 15-54 (men)
# We keep 15-80 to allow for some older respondents

print(f"Age statistics before cleaning:")
print(df['age'].describe())

before = len(df)
df = df[(df['age'] >= 15) & (df['age'] <= 80)]
after = len(df)

print(f"\n✅ Age outliers removed")
print(f"   Rows before : {before:,}")
print(f"   Rows after  : {after:,}")
print(f"   Removed     : {before - after:,} rows")

Age statistics before cleaning:
count   28494.00
mean       30.11
std        10.16
min        15.00
25%        21.00
50%        29.00
75%        38.00
max        54.00
Name: age, dtype: float64

✅ Age outliers removed
   Rows before : 28,494
   Rows after  : 28,494
   Removed     : 0 rows


In [18]:
# ── Step 9: Final summary of clean dataset ───────────────────────────────
print("=" * 50)
print("   CHRONICROISK KE — CLEAN DATASET SUMMARY")
print("=" * 50)
print(f"\n Total rows         : {len(df):,}")
print(f" Total columns      : {len(df.columns)}")
print(f"\n Missing values     : {df.isnull().sum().sum()}")
print(f" Duplicate rows     : {df.duplicated().sum()}")
print(f"\n Hypertension cases : {df['hypertension'].sum():,} ({df['hypertension'].mean()*100:.1f}%)")
print(f" Diabetes cases     : {df['diabetes'].sum():,} ({df['diabetes'].mean()*100:.1f}%)")
print(f"\n Gender breakdown:")
print(df['gender'].value_counts())
print(f"\n BMI range          : {df['bmi'].min():.1f} → {df['bmi'].max():.1f}")
print(f" Age range          : {df['age'].min()} → {df['age'].max()}")
print("=" * 50)

   CHRONICROISK KE — CLEAN DATASET SUMMARY

 Total rows         : 28,494
 Total columns      : 11

 Missing values     : 0
 Duplicate rows     : 0

 Hypertension cases : 1,800 (6.3%)
 Diabetes cases     : 272 (1.0%)

 Gender breakdown:
gender
Female    16890
Male      11604
Name: count, dtype: int64

 BMI range          : 12.0 → 59.9
 Age range          : 15 → 54


In [19]:
# ── Step 10: Save the final cleaned dataset ──────────────────────────────
df.to_csv('ChronicRiskKE_final.csv', index=False)

print("✅ Final dataset saved as: ChronicRiskKE_final.csv")
print(f"   Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print("\nReady for Phase 2 — Exploratory Data Analysis")

✅ Final dataset saved as: ChronicRiskKE_final.csv
   Shape: 28,494 rows × 11 columns

Ready for Phase 2 — Exploratory Data Analysis


## ✅ Phase 1 Complete — Data Cleaning Summary

| Step | Action | Result |
|---|---|---|
| 1 | Drop rows with missing targets | 15,255 rows removed |
| 2 | Drop unnecessary columns | weight_kg, height_cm, heart_disease removed |
| 3 | Fill missing BMI with median | 0 missing values remaining |
| 4 | Fill missing smoking with Unknown | 0 missing values remaining |
| 5 | Fix data types | hypertension, diabetes → int |
| 6 | Remove duplicates | 0 found |
| 7 | Remove BMI outliers (< 10 or > 60) | Extreme values removed |
| 8 | Remove age outliers (< 15 or > 80) | Clean age range: 15–54 |

**Final dataset: 28,494 rows × 11 columns — 0 missing values**

➡️ Next: `02_eda.ipynb` — Exploratory Data Analysis